# 03. Model Comparison

Trains and evaluates Logistic Regression, Random Forest, and Gradient Boosting.

In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from resume_scanner.ml.dataset import ResumeJDDatasetGenerator, create_train_test_split
from resume_scanner.ml.features import FeatureExtractor
from resume_scanner.ml.model import ModelTrainer, save_model
from resume_scanner.ml.embeddings import get_encoder

In [ ]:
# 1. Generate Dataset
gen = ResumeJDDatasetGenerator(seed=42)
entries = gen.generate(pairs_per_domain=100)
train_entries, test_entries = create_train_test_split(entries)

print(f'Train: {len(train_entries)}, Test: {len(test_entries)}')

In [ ]:
# 2. Extract Features
encoder = get_encoder()
extractor = FeatureExtractor(encoder=encoder)

def process(entries):
    X, y = [], []
    for e in entries:
        X.append(extractor.extract_vector(e.resume_text, e.job_description))
        y.append(e.match_label)
    return np.array(X), np.array(y)

X_train, y_train = process(train_entries)
X_test, y_test = process(test_entries)

In [ ]:
# 3. Train & Compare
trainer = ModelTrainer(feature_names=extractor.extract_vector.__code__.co_varnames) # Mock feature names for display
results = trainer.train_and_compare(X_train, y_train, X_test, y_test)

import json
print(json.dumps(results['comparison'], indent=2))

In [ ]:
# 4. Save best model
from resume_scanner.ml.features import FEATURE_SCHEMA
save_model(trainer.best_model, trainer.scaler, results['best_metrics'], FEATURE_SCHEMA)